# Book Editions, Popularity and Reader Ratings

## Introduction

This project explores the relationship between the number of
catalogued editions, reader popularity and average book ratings.

It combines Goodreads data distributed through Kaggle with
catalogue information from Open Library.

Popularity is represented by the number of Goodreads ratings.
The number of editions is measured using Open Library catalogue data.

The analysis investigates associations. It does not establish
that more editions cause greater popularity or higher ratings.

## Research Questions

1. Is the number of catalogued editions associated with the
   number of Goodreads ratings?

2. Is the number of Goodreads ratings associated with the
   average reader rating?

3. How sensitive are the findings to low rating counts,
   the choice of a representative edition and extreme observations?

A possible additional analysis will compare mean reader ratings
between groups defined by edition count. The grouping rule and
test assumptions will be documented before performing the test.

## Data Sources

### Goodreads data distributed through Kaggle

Source:
https://www.kaggle.com/datasets/jealousleopard/goodreadsbooks

Main variables:
- `isbn13`: identifier used to match editions;
- `title`: book title;
- `authors`: author and contributor information;
- `average_rating`: average rating reported in the dataset;
- `ratings_count`: number of ratings reported in the dataset.

### Open Library

Source:
https://openlibrary.org/

API documentation:
https://openlibrary.org/dev/docs/api/search

Main variables:
- `isbn13`: identifier used for matching;
- `openlibrary_work_id`: identifier of the literary work;
- `edition_count`: number of editions catalogued for that work;
- `retrieved_on`: date of data retrieval.

The saved Open Library pilot responses were retrieved on
20 November 2026. The Goodreads snapshot date has not yet
been established.

The two sources describe different systems and may represent
different points in time.

## Unit of Analysis and Pilot Scope

The final analysis will use one row per literary work.

An ISBN identifies an edition, while an Open Library work ID
groups editions of a work. Matching will therefore start with
ISBNs, followed by work-level validation and deduplication.

When several Goodreads records map to the same work, a documented
rule will select a representative record. Rating counts from
different editions will not be summed without evidence that
they are non-overlapping.

The initial pilot contains five deliberately selected books.
Its purpose is to validate loading, matching and saving data.

The pilot is not a representative sample and will not be used
for statistical conclusions.

A larger dataset will be prepared before the main analysis.

## Pilot Data Integration

This section checks the data integration workflow using five books.

The steps are:
1. Load the Goodreads and Open Library tables.
2. Check identifiers and data types.
3. Merge the tables by ISBN-13.
4. Inspect matching results and missing values.
5. Save the merged pilot dataset.

These steps validate data handling. The five deliberately selected
books will not be used for statistical conclusions.

In [1]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
kaggle = pd.read_csv(
    RAW_DIR / "kaggle_pilot.csv",
    dtype={
        "isbn": "string",
        "isbn13": "string",
    },
)

openlibrary = pd.read_csv(
    RAW_DIR / "openlibrary_pilot.csv",
    dtype={
        "isbn13": "string",
        "openlibrary_work_id": "string",
    },
)

In [3]:
kaggle[
    [
        "title",
        "authors",
        "isbn13",
        "average_rating",
        "ratings_count",
    ]
]

,title,authors,isbn13,average_rating,ratings_count
0,Pride and Prejudice,Jane Austen,9780192802385,4.26,2399
1,Great Expectations,Charles Dickens/David Trotter/Charlotte Mitchell,9780141439563,3.77,14228
2,Wuthering Heights,Emily Brontë,9780140620122,3.85,2155
3,Jane Eyre,Charlotte Brontë/Michael Mason,9780142437209,4.12,1409369
4,Dracula,Bram Stoker,9780140620634,3.99,1683


In [4]:
openlibrary[
    [
        "openlibrary_title",
        "openlibrary_authors",
        "isbn13",
        "openlibrary_work_id",
        "edition_count",
    ]
]

,openlibrary_title,openlibrary_authors,isbn13,openlibrary_work_id,edition_count
0,Pride and Prejudice,Jane Austen,9780192802385,OL66554W,4042
1,Great Expectations,Charles Dickens,9780141439563,OL8721462W,1489
2,Wuthering Heights,Emily Brontë,9780140620122,OL21177W,2889
3,Jane Eyre,Charlotte Brontë,9780142437209,OL1095427W,1170
4,Dracula,Bram Stoker,9780140620634,OL85892W,1920


In [5]:
kaggle.info()
openlibrary.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              5 non-null      int64  
 1   title               5 non-null      str    
 2   authors             5 non-null      str    
 3   average_rating      5 non-null      float64
 4   isbn                5 non-null      string 
 5   isbn13              5 non-null      string 
 6   language_code       5 non-null      str    
 7   num_pages           5 non-null      int64  
 8   ratings_count       5 non-null      int64  
 9   text_reviews_count  5 non-null      int64  
 10  publication_date    5 non-null      str    
 11  publisher           5 non-null      str    
dtypes: float64(1), int64(4), str(5), string(2)
memory usage: 1.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------  

In [6]:
key_checks = pd.DataFrame(
    {
        "source": ["Kaggle / Goodreads", "Open Library"],
        "rows": [
            len(kaggle),
            len(openlibrary),
        ],
        "missing_isbn13": [
            kaggle["isbn13"].isna().sum(),
            openlibrary["isbn13"].isna().sum(),
        ],
        "duplicate_isbn13": [
            kaggle["isbn13"].duplicated().sum(),
            openlibrary["isbn13"].duplicated().sum(),
        ],
    }
)

key_checks

,source,rows,missing_isbn13,duplicate_isbn13
0,Kaggle / Goodreads,5,0,0
1,Open Library,5,0,0


### Matching Strategy

The tables are matched by ISBN-13.

A left join preserves every Goodreads record and makes unmatched
records visible.

The pilot expects one record per ISBN in each input table,
so the merge uses one-to-one validation.

Titles and author information will also be inspected to check
whether the matched records refer to the same literary work.

In [7]:
assert kaggle["isbn13"].notna().all(), "Missing ISBN in Kaggle data"
assert openlibrary["isbn13"].notna().all(), "Missing ISBN in Open Library data"

books_pilot = kaggle.merge(
    openlibrary,
    on="isbn13",
    how="left",
    validate="one_to_one",
    indicator=True,
)

books_pilot[
    [
        "title",
        "isbn13",
        "average_rating",
        "ratings_count",
        "edition_count",
        "openlibrary_work_id",
        "_merge",
    ]
]

,title,isbn13,average_rating,ratings_count,edition_count,openlibrary_work_id,_merge
0,Pride and Prejudice,9780192802385,4.26,2399,4042,OL66554W,both
1,Great Expectations,9780141439563,3.77,14228,1489,OL8721462W,both
2,Wuthering Heights,9780140620122,3.85,2155,2889,OL21177W,both
3,Jane Eyre,9780142437209,4.12,1409369,1170,OL1095427W,both
4,Dracula,9780140620634,3.99,1683,1920,OL85892W,both


In [8]:
merge_summary = pd.DataFrame(
    {
        "check": [
            "Goodreads rows before merge",
            "Rows after merge",
            "Matched rows",
            "Unmatched Goodreads rows",
            "Unique matched works",
        ],
        "value": [
            len(kaggle),
            len(books_pilot),
            books_pilot["_merge"].eq("both").sum(),
            books_pilot["_merge"].eq("left_only").sum(),
            books_pilot["openlibrary_work_id"].nunique(),
        ],
    }
)

merge_summary

,check,value
0,Goodreads rows before merge,5
1,Rows after merge,5
2,Matched rows,5
3,Unmatched Goodreads rows,0
4,Unique matched works,5


In [9]:
books_pilot[
    [
        "title",
        "openlibrary_title",
        "authors",
        "openlibrary_authors",
    ]
]

,title,openlibrary_title,authors,openlibrary_authors
0,Pride and Prejudice,Pride and Prejudice,Jane Austen,Jane Austen
1,Great Expectations,Great Expectations,Charles Dickens/David Trotter/Charlotte Mitchell,Charles Dickens
2,Wuthering Heights,Wuthering Heights,Emily Brontë,Emily Brontë
3,Jane Eyre,Jane Eyre,Charlotte Brontë/Michael Mason,Charlotte Brontë
4,Dracula,Dracula,Bram Stoker,Bram Stoker


In [10]:
analysis_columns = [
    "average_rating",
    "ratings_count",
    "edition_count",
]

books_pilot[analysis_columns].isna().sum()

average_rating    0
ratings_count     0
edition_count     0
dtype: int64

In [11]:
books_pilot[
    [
        "title",
        "first_publish_year_raw",
        "notes",
    ]
]

,title,first_publish_year_raw,notes
0,Pride and Prejudice,1813,NaN
1,Great Expectations,1861,NaN
2,Wuthering Heights,1846,Year 1846 requires bibliographic verification ...
3,Jane Eyre,1847,NaN
4,Dracula,1897,NaN


In [12]:
output_path = PROCESSED_DIR / "books_pilot_merged.csv"

books_pilot.to_csv(
    output_path,
    index=False,
)

print(f"Saved {len(books_pilot)} rows to {output_path}")

Saved 5 rows to data\processed\books_pilot_merged.csv


### Pilot Outcome

Both input tables contained five records with complete and
unique ISBN-13 identifiers.

All five Goodreads records matched Open Library records.
The merge preserved the original row count and produced
five distinct Open Library work IDs.

Titles and author information were inspected for consistency.
No missing values were found in the three main analysis variables:
average rating, rating count and edition count.

A potentially unreliable publication-year value was retained
with its source note and excluded from the planned analysis.

The merged pilot dataset was saved to
`data/processed/books_pilot_merged.csv`.

The pilot confirms that the integration workflow works for
these records. It does not establish representativeness,
validate the full source dataset or support statistical conclusions.